In [2]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
CSV_PATH = "../data/price_prediction/iphone_realistic_market.csv"
MODEL_SAVE_DIR = "../models/price_prediction"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print("CSV exists:", os.path.exists(CSV_PATH))
print("Model folder exists:", os.path.exists(MODEL_SAVE_DIR))

CSV exists: True
Model folder exists: True


In [4]:
df = pd.read_csv(CSV_PATH)
print(df.shape)
df.head()

(10000, 5)


,Model,Storage_GB,Battery_Health,Condition,Price_USD
0,iPhone-13-Pro,512,85,Used,821
1,iPhone-14,128,100,Used,738
2,iPhone-13-Pro,1024,85,Used,863
3,iPhone-7,64,94,Used,231
4,iPhone-1st-gen-,16,83,Used,433


In [5]:
print(df.columns.tolist())
print(df.info())

['Model', 'Storage_GB', 'Battery_Health', 'Condition', 'Price_USD']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Model           10000 non-null  object
 1   Storage_GB      10000 non-null  int64 
 2   Battery_Health  10000 non-null  int64 
 3   Condition       10000 non-null  object
 4   Price_USD       10000 non-null  int64 
dtypes: int64(3), object(2)
memory usage: 390.8+ KB
None


In [6]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Model', 'Storage_GB', 'Battery_Health', 'Condition', 'Price_USD']


In [7]:
df = df[["Model", "Storage_GB", "Battery_Health", "Condition", "Price_USD"]].copy()
df = df.dropna()
print(df.shape)
df.head()

(10000, 5)


,Model,Storage_GB,Battery_Health,Condition,Price_USD
0,iPhone-13-Pro,512,85,Used,821
1,iPhone-14,128,100,Used,738
2,iPhone-13-Pro,1024,85,Used,863
3,iPhone-7,64,94,Used,231
4,iPhone-1st-gen-,16,83,Used,433


In [8]:
df["Storage_GB"] = pd.to_numeric(df["Storage_GB"], errors="coerce")
df["Battery_Health"] = pd.to_numeric(df["Battery_Health"], errors="coerce")
df["Price_USD"] = pd.to_numeric(df["Price_USD"], errors="coerce")
df["Condition"] = df["Condition"].astype(str).str.strip().str.lower()
df["Model"] = df["Model"].astype(str).str.strip()
df = df.dropna()
print(df.shape)
df.head()

(10000, 5)


,Model,Storage_GB,Battery_Health,Condition,Price_USD
0,iPhone-13-Pro,512,85,used,821
1,iPhone-14,128,100,used,738
2,iPhone-13-Pro,1024,85,used,863
3,iPhone-7,64,94,used,231
4,iPhone-1st-gen-,16,83,used,433


In [9]:
X = df[["Model", "Storage_GB", "Battery_Health", "Condition"]].copy()
y = df["Price_USD"].copy()
print(X.head())
print(y.head())

             Model  Storage_GB  Battery_Health Condition
0    iPhone-13-Pro         512              85      used
1        iPhone-14         128             100      used
2    iPhone-13-Pro        1024              85      used
3         iPhone-7          64              94      used
4  iPhone-1st-gen-          16              83      used
0    821
1    738
2    863
3    231
4    433
Name: Price_USD, dtype: int64


In [10]:
le_model = LabelEncoder()
le_condition = LabelEncoder()
X["Model"] = le_model.fit_transform(X["Model"])
X["Condition"] = le_condition.fit_transform(X["Condition"])
print(X.head())
print("Models:", list(le_model.classes_))
print("Conditions:", list(le_condition.classes_))

   Model  Storage_GB  Battery_Health  Condition
0      6         512              85          0
1      7         128             100          0
2      6        1024              85          0
3     11          64              94          0
4     10          16              83          0
Models: ['iPhone 15 Pro', 'iPhone-11', 'iPhone-11-Pro', 'iPhone-12', 'iPhone-12-Pro', 'iPhone-13', 'iPhone-13-Pro', 'iPhone-14', 'iPhone-14-Pro', 'iPhone-15', 'iPhone-1st-gen-', 'iPhone-7', 'iPhone-8', 'iPhone-SE', 'iPhone-Xr', 'iPhone-Xs']
Conditions: ['used']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2, random_state=42
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (8000, 4)
Test shape: (2000, 4)


In [12]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
dt_mae = mean_absolute_error(y_test, dt_preds)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_preds))
dt_r2 = r2_score(y_test, dt_preds)
print("Decision Tree Results")
print("MAE:", dt_mae)
print("RMSE:", dt_rmse)
print("R2:", dt_r2)

Decision Tree Results
MAE: 16.63313789127539
RMSE: 19.778327838080916
R2: 0.9933521247497046


In [13]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)
print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Results
MAE: 16.401225765151707
RMSE: 19.407612360536717
R2: 0.9935989983827769


In [14]:
results_df = pd.DataFrame({
   "Model": ["Decision Tree", "Random Forest"],
   "MAE": [dt_mae, rf_mae],
   "RMSE": [dt_rmse, rf_rmse],
   "R2": [dt_r2, rf_r2]
})
results_df

,Model,MAE,RMSE,R2
0,Decision Tree,16.633138,19.778328,0.993352
1,Random Forest,16.401226,19.407612,0.993599


In [15]:
joblib.dump(rf, os.path.join(MODEL_SAVE_DIR, "best_price_model.pkl"))
joblib.dump(le_model, os.path.join(MODEL_SAVE_DIR, "label_encoder_model.pkl"))
joblib.dump(le_condition, os.path.join(MODEL_SAVE_DIR, "label_encoder_condition.pkl"))
print("Saved:")
print("- best_price_model.pkl")
print("- label_encoder_model.pkl")
print("- label_encoder_condition.pkl")

Saved:
- best_price_model.pkl
- label_encoder_model.pkl
- label_encoder_condition.pkl


In [16]:
def predict_price(model_name, storage_gb, battery_health, condition):
   if model_name not in le_model.classes_:
       raise ValueError(f"Model '{model_name}' not found. Available models: {list(le_model.classes_)}")
   if condition not in le_condition.classes_:
       raise ValueError(f"Condition '{condition}' not found. Available conditions: {list(le_condition.classes_)}")
   sample_df = pd.DataFrame([{
       "Model": model_name,
       "Storage_GB": storage_gb,
       "Battery_Health": battery_health,
       "Condition": condition
   }])
   sample_df["Model"] = le_model.transform(sample_df["Model"])
   sample_df["Condition"] = le_condition.transform(sample_df["Condition"])
   return rf.predict(sample_df)[0]

In [17]:
print("Available models:")
print(list(le_model.classes_))

Available models:
['iPhone 15 Pro', 'iPhone-11', 'iPhone-11-Pro', 'iPhone-12', 'iPhone-12-Pro', 'iPhone-13', 'iPhone-13-Pro', 'iPhone-14', 'iPhone-14-Pro', 'iPhone-15', 'iPhone-1st-gen-', 'iPhone-7', 'iPhone-8', 'iPhone-SE', 'iPhone-Xr', 'iPhone-Xs']


In [18]:
price = predict_price("iPhone-14", 128, 90, "used")
print(f"Predicted price: ${price:.2f}")

Predicted price: $684.36
